# 03 · Conversion Models

Booking and sell-through classifiers with calibration and lift.

> Synthetic data — methodology demo, not CARS24 results. Run `make pipeline-small` first so artifacts exist.

In [ ]:
from driveintent.config import load_config
from driveintent.models import classifiers
cfg = load_config()
for which in ['booking','sellthrough']:
    m = classifiers.train(cfg, which)['catboost_calibrated']
    print(which, 'PR-AUC', round(m['pr_auc'],3), 'ROC', round(m['roc_auc'],3),
          'Brier', round(m['brier'],4), 'ECE', round(m['ece'],3))

## Reliability & lift (booking)

In [ ]:
import pandas as pd, plotly.express as px
rel = pd.read_csv(cfg.artifacts/'metrics'/'booking_reliability.csv')
lift = pd.read_csv(cfg.artifacts/'metrics'/'booking_lift_table.csv')
px.line(rel, x='mean_predicted', y='observed_rate', markers=True,
        title='Booking reliability').show()
px.bar(lift, x='decile', y='lift', title='Booking lift by decile').show()

## Explain a lead

In [ ]:
ds = pd.read_parquet(cfg.processed_data/'booking_dataset.parquet').head(1)
print('P(book)=', float(classifiers.predict_proba(cfg,'booking',ds)[0]))
classifiers.explain(cfg, 'booking', ds, top_k=5)